# Веда — живые кадры
Меню **Среда выполнения → Выполнить всё** (Ctrl+F9). Через ~5 минут скачается `veda_frames.zip` — пришлите его в чат.

In [ ]:
#@title 1. Подготовка (1–2 минуты)
import torch, sys, subprocess, os
if not torch.cuda.is_available():
    raise RuntimeError('Нет видеокарты. Меню: Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить, затем снова «Выполнить всё».')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dill', 'ultralytics'], check=True)
if not os.path.exists('/content/advanced_live_portrait'):
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/PowerHouseMan/ComfyUI-AdvancedLivePortrait.git', '/content/advanced_live_portrait'], check=True)
print('Готово')

In [ ]:
#@title 2. Загрузка нейросети LivePortrait
import os, sys, types, torch, safetensors.torch
os.chdir('/content'); sys.path.insert(0, '/content')
MODELS = '/content/models'; TEMP = '/content/temp'
for d in (MODELS + '/liveportrait', MODELS + '/ultralytics', TEMP, '/content/outputs'): os.makedirs(d, exist_ok=True)
fp = types.ModuleType('folder_paths')
fp.models_dir = MODELS; fp.output_directory = '/content/outputs'
fp.get_folder_paths = lambda m: [os.path.join(MODELS, m)]
fp.get_save_image_path = lambda f, d, *a, **k: (d, f, 0, '', f)
fp.get_temp_directory = lambda: TEMP
fp.add_model_folder_path = lambda *a, **k: None
sys.modules['folder_paths'] = fp
comfy = types.ModuleType('comfy'); cu = types.ModuleType('comfy.utils')
def _load(ckpt, *a, **k):
    s = str(ckpt)
    return safetensors.torch.load_file(s) if s.endswith('.safetensors') else torch.load(s, map_location='cpu', weights_only=False)
class _PB:
    def __init__(self, *a, **k): pass
    def update(self, *a, **k): pass
    def update_absolute(self, *a, **k): pass
cu.load_torch_file = _load; cu.ProgressBar = _PB; comfy.utils = cu
sys.modules['comfy'] = comfy; sys.modules['comfy.utils'] = cu
from advanced_live_portrait.nodes import ExpressionEditor, g_engine
g_engine.get_pipeline(); g_engine.get_detect_model()
print('Нейросеть загружена')

In [ ]:
#@title 3. Генерация кадров Веды (2–5 минут)
import base64, io, numpy as np, zipfile, json
from PIL import Image
SRC_B64 = "..."
src = Image.open(io.BytesIO(base64.b64decode(SRC_B64))).convert('RGB')
src_t = torch.from_numpy(np.array(src, dtype=np.float32) / 255.0).unsqueeze(0)
ed = ExpressionEditor()
def render(yaw=0, pitch=0, px=0, py=0, blink=0):
    out = ed.run(rotate_pitch=float(pitch), rotate_yaw=float(yaw), rotate_roll=0.0, blink=float(blink), eyebrow=0.0, wink=0.0,
                 pupil_x=float(px), pupil_y=float(py), aaa=0.0, eee=0.0, woo=0.0, smile=0.0, src_ratio=1.0, sample_ratio=1.0,
                 sample_parts='OnlyExpression', crop_factor=1.7, src_image=src_t)
    t = out['result'][0]
    return Image.fromarray((t.squeeze(0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8))
YAWS = list(range(-12, 13, 2))      # 13 положений влево-вправо
PITCHES = list(range(-8, 9, 2))     # 9 положений вверх-вниз
os.makedirs('/content/veda', exist_ok=True)
meta = {'yaws': YAWS, 'pitches': PITCHES, 'pupil_x_per_yaw': 1.25, 'pupil_y_per_pitch': -1.5, 'blinks': [-5, -10, -15, -20]}
n = 0; total = len(YAWS) * len(PITCHES) + 4
for pi, p in enumerate(PITCHES):
    for yi, y in enumerate(YAWS):
        render(y, p, y * 1.25, -p * 1.5).save(f'/content/veda/g_p{pi:02d}_y{yi:02d}.webp', quality=92, method=6)
        n += 1
    print(f'{n}/{total}')
for i, b in enumerate(meta['blinks']):
    render(blink=b).save(f'/content/veda/blink_{i}.webp', quality=92, method=6)
json.dump(meta, open('/content/veda/meta.json', 'w'))
# контрольный лист: все кадры мелко
th = 96; sheet = Image.new('RGB', (th * len(YAWS), th * len(PITCHES)))
for pi in range(len(PITCHES)):
    for yi in range(len(YAWS)):
        sheet.paste(Image.open(f'/content/veda/g_p{pi:02d}_y{yi:02d}.webp').resize((th, th)), (yi * th, pi * th))
sheet.save('/content/veda/_sheet.jpg', quality=85)
with zipfile.ZipFile('/content/veda_frames.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in sorted(os.listdir('/content/veda')): z.write('/content/veda/' + f, f)
print('Готово:', n + 4, 'кадров')
display(sheet)

In [ ]:
#@title 4. Скачать архив (пришлите его в чат Claude)
from google.colab import files
files.download('/content/veda_frames.zip')